In [1]:
import numpy as np
import scipy.sparse as sp

## Step 1: Theoretical Understanding of PageRank
Learning PageRank and the power method by analyzing the example graph in the textbook.

In [2]:
def load_graph(data_file_path):
    edges = []
    node_map = {}
    node_counter = 0
    with open(data_file_path, 'r') as f:
        for line in f:
            if not line.startswith('#'):
                parts = line.strip().split('\t')
                u, v = int(parts[0]), int(parts[1])
                if u not in node_map:
                    node_map[u] = node_counter
                    node_counter += 1
                if v not in node_map:
                    node_map[v] = node_counter
                    node_counter += 1
                edges.append((node_map[u], node_map[v]))
    edges = np.array(edges)
    n_nodes = node_counter
    return edges, n_nodes

edges, n_nodes = load_graph('../../data/Textbook-Example.txt')
print(f"Edges:\n{edges}\n")
print(f"Number of nodes:\n{n_nodes}")

Edges:
[[0 1]
 [1 2]
 [1 3]
 [2 3]
 [2 4]
 [2 5]
 [3 0]
 [4 5]
 [5 0]]

Number of nodes:
6


In [3]:
# A simple implementation of the matrix A according to the textbook.
# Use CSR format in the future for better performance.
def build_adjacency_matrix(edges, n_nodes, p):
    G = np.zeros((n_nodes, n_nodes))
    for i, j in edges:
        G[j, i] = 1
    out_degrees = G.sum(axis=0)
    D_diag = np.zeros(n_nodes)
    non_zero_mask = out_degrees > 0
    D_diag[non_zero_mask] = 1.0 / out_degrees[non_zero_mask]
    D = np.diag(D_diag)
    e = np.ones(n_nodes)
    f = np.full(n_nodes, 1.0 / n_nodes)
    f[non_zero_mask] = (1 - p) / n_nodes
    A = p * G @ D + np.outer(e, f)
    print(f"Matrix G:\n{G}\n")
    print(f"Matrix D:\n{D}\n")
    print(f"Vector e:\n{e}\n")
    print(f"Vector f:\n{f}\n")
    return A

A = build_adjacency_matrix(edges, n_nodes, 0.85)
print(f"Matrix A:\n{A}")

Matrix G:
[[0. 0. 0. 1. 0. 1.]
 [1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0.]
 [0. 1. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 1. 0.]]

Matrix D:
[[1.         0.         0.         0.         0.         0.        ]
 [0.         0.5        0.         0.         0.         0.        ]
 [0.         0.         0.33333333 0.         0.         0.        ]
 [0.         0.         0.         1.         0.         0.        ]
 [0.         0.         0.         0.         1.         0.        ]
 [0.         0.         0.         0.         0.         1.        ]]

Vector e:
[1. 1. 1. 1. 1. 1.]

Vector f:
[0.025 0.025 0.025 0.025 0.025 0.025]

Matrix A:
[[0.025      0.025      0.025      0.875      0.025      0.875     ]
 [0.875      0.025      0.025      0.025      0.025      0.025     ]
 [0.025      0.45       0.025      0.025      0.025      0.025     ]
 [0.025      0.45       0.30833333 0.025      0.025      0.025     ]
 [0.025      0.025      0.30833333 0.025      0.025      0.025     ]
 

In [4]:
def build_components(edges, n_nodes, p):
    cols = edges[:, 0]
    rows = edges[:, 1]
    data = np.ones(len(edges))
    G = sp.csr_matrix((data, (rows, cols)), shape=(n_nodes, n_nodes))
    out_degrees = np.array(G.sum(axis=0)).flatten()
    D = np.zeros(n_nodes)
    non_zero_mask = out_degrees > 0
    D[non_zero_mask] = 1.0 / out_degrees[non_zero_mask]
    e = np.ones(n_nodes)
    f = np.zeros(n_nodes)
    for i in range(n_nodes):
        if out_degrees[i] != 0:
            f[i] = (1-p)/n_nodes
        else:
            f[i] = 1/n_nodes
    v = e * 1/n_nodes
    return G, D, e, f, v

G, D, e, f, v = build_components(edges, n_nodes, 0.85)
print(f"Matrix G:\n{G.toarray()}\n")
print(f"Vector D:\n{D}\n")
print(f"Vector e:\n{e}\n")
print(f"Vector f:\n{f}\n")
print(f"Vector v:\n{v}\n")

Matrix G:
[[0. 0. 0. 1. 0. 1.]
 [1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0.]
 [0. 1. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 1. 0.]]

Vector D:
[1.         0.5        0.33333333 1.         1.         1.        ]

Vector e:
[1. 1. 1. 1. 1. 1.]

Vector f:
[0.025 0.025 0.025 0.025 0.025 0.025]

Vector v:
[0.16666667 0.16666667 0.16666667 0.16666667 0.16666667 0.16666667]



In [5]:
def power_method_A(A, n_nodes, max_iter=100, tol=1e-6):
    x = np.random.rand(n_nodes)
    x /= np.linalg.norm(x, 1)
    for _ in range(max_iter):
        x_new = A @ x
        x_new /= np.linalg.norm(x_new, 1)
        if np.linalg.norm(x_new - x, 1) < tol:
            break
        x = x_new
    return x

pagerank = power_method_A(A, n_nodes)
print(f"PageRank:\n{pagerank}")

PageRank:
[0.26752827 0.252399   0.13226933 0.16974567 0.06247635 0.11558138]


In [6]:
def power_method_components(G, D, e, f, n_nodes, p, max_iter=100, tol=1e-6):
    x = np.random.rand(n_nodes)
    x /= np.linalg.norm(x, 1)
    for _ in range(max_iter):
        x_new = p * G @ (D * x) + e * (f @ x)
        x_new /= np.linalg.norm(x_new, 1)
        if np.linalg.norm(x_new - x, 1) < tol:
            break
        x = x_new
    return x

pagerank = power_method_components(G, D, e, f, n_nodes, 0.85)
print(f"PageRank:\n{pagerank}")

PageRank:
[0.26752795 0.25239868 0.13226968 0.16974608 0.06247641 0.11558121]


In [8]:
def power_method_standard(G, D, e, v, n_nodes, p, max_iter=100, tol=1e-6):
    x = np.random.rand(n_nodes)
    x /= np.linalg.norm(x, 1)
    for _ in range(max_iter):
        x_new = p * G @ (D * x) + (1-p) * e * (v @ x)
        x_new /= np.linalg.norm(x_new, 1)
        if np.linalg.norm(x_new - x, 1) < tol:
            break
        x = x_new
    return x

pagerank = power_method_standard(G, D, e, v, n_nodes, 0.85)
print(f"PageRank:\n{pagerank}")

PageRank:
[0.26752824 0.25239887 0.13226938 0.16974576 0.06247638 0.11558137]
